# Indian Traffic Sign Recognition (TSR) - Stage 2: EfficientNet-B0 Classifier
### Fine-Grained Classification Pipeline for Cropped Signs (Google Colab GPU)

This notebook trains a lightweight **EfficientNet-B0** classifier on cropped sign patches (224x224).

**Roadmap:**
1. Setup PyTorch GPU environment
2. Prepare cropped sign dataset from YOLO bounding boxes
3. Apply Albumentations (rain, glare, blur, shadows)
4. Fine-tune EfficientNet-B0 with Cross-Entropy Loss & AdamW
5. Evaluate Accuracy, Precision, Recall, and F1-score
6. Save and export `efficientnet_tsr.pth`

In [ ]:
# Step 1: Install requirements
!pip install --upgrade torchvision albumentations scikit-learn seaborn matplotlib

In [ ]:
# Step 2: GPU check & Imports
import torch
import torch.nn as nn
import torchvision
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
import os, sys, glob
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Step 3: Define Crop Dataset Loader from YOLO annotations
class CroppedTSRDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.samples = []
        self.transform = transform
        img_dir = os.path.join(split_dir, "images")
        lbl_dir = os.path.join(split_dir, "labels")
        
        for img_path in glob.glob(os.path.join(img_dir, "*.*")):
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(lbl_dir, f"{base_name}.txt")
            if not os.path.exists(lbl_path):
                continue
            
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cid, xc, yc, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                        self.samples.append((img_path, cid, xc, yc, w, h))
        print(f"Loaded {len(self.samples)} sign crops from {split_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, cid, xc, yc, w, h = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        iw, ih = img.size
        
        x1 = max(0, int((xc - w / 2) * iw))
        y1 = max(0, int((yc - h / 2) * ih))
        x2 = min(iw, int((xc + w / 2) * iw))
        y2 = min(ih, int((yc + h / 2) * ih))
        
        crop = img.crop((x1, y1, x2, y2))
        if self.transform:
            crop = self.transform(crop)
        return crop, cid

In [ ]:
# Step 4: Transforms and DataLoaders
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = CroppedTSRDataset("/content/dataset/train", transform=train_transform)
val_dataset = CroppedTSRDataset("/content/dataset/valid", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Step 5: Model Definition (EfficientNet-B0)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 15)  # 15 Indian TSR classes
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

In [ ]:
# Step 6: Training Loop
epochs = 30
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    scheduler.step()
    train_acc = 100.0 * correct / max(1, total)
    
    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100.0 * val_correct / max(1, val_total)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/max(1, total):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "efficientnet_tsr.pth")
        print(f"--> Saved new best model checkpoint (Val Acc: {best_val_acc:.2f}%)")

In [ ]:
# Step 7: Export model to Google Drive
import shutil
from google.colab import files

if os.path.exists("efficientnet_tsr.pth"):
    os.makedirs("/content/drive/MyDrive/TSR_Models", exist_ok=True)
    shutil.copy("efficientnet_tsr.pth", "/content/drive/MyDrive/TSR_Models/efficientnet_tsr.pth")
    print("Saved classifier weights to Google Drive!")
    files.download("efficientnet_tsr.pth")